# 05. マルチターン Chat と Function Calling（ツール連携）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/google-genai/ja/05_chat_and_function_calling.ipynb)

`google-genai` SDK は、ステートレスなモデル呼び出しを「状態とツールを備えた実用的なアシスタント」へと進化させる2つの標準プリミティブを提供しています。会話履歴を自動管理する **`client.chats.create()`** と、Python の型ヒントおよび docstring からスキーマを自動抽出し、ローカル関数の実行から結果の再投入までを自動化する **自動 Function Calling（`tools=[fn]`）** です。

![図5-1: google-genai SDK における Python 関数の自動 Function Calling ループ](https://examples.drawlib.com/google-genai/images/05_chat_and_function_calling/ch05_function_calling_loop_ja.png)

*図5-1: google-genai SDK における Python 関数の自動 Function Calling ループ*

---

## 1. 3行のクライアント初期化とマルチターン会話（`client.chats.create`）

Google Colab のシークレット（`userdata.get`）から `GEMINI_API_KEY` を読み込み、共通の `genai.Client` を3行で初期化します。

In [ ]:
!pip install -q drawlib google-genai pydantic

In [ ]:
from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

単発の `client.models.generate_content()` とは異なり、**`client.chats.create()`** はセッション内部で `list[types.Content]` 形式の会話履歴を保持します。`chat.send_message()` を呼ぶたびに `Content(role="user")` とモデルの応答 `Content(role="model")`（および途中のツール呼び出し履歴）が自動的に追記されます。さらにマルチターン会話が長くなると、毎ターン送信される共通プレフィックスに対して Gemini 2.5 の **暗黙的コンテキストキャッシュ（Implicit Context Caching）** が追加設定なしで自動的に働き、レイテンシ（TTFT）と入力トークンコストが抑えられます。

![図5-2: chats.create() による会話履歴の蓄積とツール連携セッションの構造](https://examples.drawlib.com/google-genai/images/05_chat_and_function_calling/ch05_chat_state_context_ja.png)

*図5-2: chats.create() による会話履歴の蓄積とツール連携セッションの構造*

`gemini-2.5-flash` でチャットセッションを開始し、2回連続でフォローアップ質問を送信したあと、`chat.get_history()` に蓄積された `user` と `model` のターンを確認してみましょう。

In [ ]:
chat = client.chats.create(model="gemini-2.5-flash")

reply_1 = chat.send_message(
    "マルチリージョン構成の Cloud Run API ゲートウェイを設計中です。重要なポイントを2点に絞って教えてください。"
)
print("Model (Turn 1):\n", reply_1.text)

reply_2 = chat.send_message(
    "その2点を踏まえると、クライアント側の HTTP タイムアウトとリトライ戦略はどう設定すべきですか？"
)
print("\nModel (Turn 2):\n", reply_2.text)

print("\n--- 蓄積されたセッション履歴 ---")
for idx, turn in enumerate(chat.get_history(), start=1):
    preview = " ".join(
        p.text.strip() for p in (turn.parts or []) if p.text and not p.thought
    )
    print(f"[{idx}] role={turn.role:<5} | {preview[:85]}...")

---

## 2. Python 関数の自動 Function Calling と Drawlib インライン可視化

外部の監視メトリクスや社内 API を Gemini に連携させるには、型ヒント（`region: str -> dict[str, float]`）と **Google スタイルの docstring**（`Args:` / `Returns:`）を記述した通常の Python 関数を **`types.GenerateContentConfig(tools=[...])`** にそのまま渡します。SDK が関数シグネチャから JSON スキーマを自動生成します。

複数リージョンの比較を求められた Gemini が `FunctionCall` を発行すると、`genai.Client` が各リージョンに対してローカルの `get_region_latency()` を自動実行し、`FunctionResponse` をモデルへ返送して実測値に基づく推奨構成をまとめ上げます。

In [ ]:
from google.genai import types

REGION_LATENCY_MS: dict[str, dict[str, float]] = {
    "asia-northeast1": {"p50_ms": 14.2, "p95_ms": 31.5},
    "us-central1": {"p50_ms": 24.5, "p95_ms": 58.4},
    "europe-west1": {"p50_ms": 38.6, "p95_ms": 84.2},
}


def get_region_latency(region: str) -> dict[str, float]:
    """指定されたクラウドリージョンの最新 API レイテンシ（p50 / p95 ミリ秒）を取得します。

    Args:
        region: 対象リージョン ID（'asia-northeast1', 'us-central1', 'europe-west1'）。

    Returns:
        'p50_ms'（中央値）と 'p95_ms'（95パーセンタイル）を含む辞書。
    """
    return REGION_LATENCY_MS.get(region, {"p50_ms": -1.0, "p95_ms": -1.0})


tool_chat = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        tools=[get_region_latency],
        temperature=0.1,
    ),
)

recommendation = tool_chat.send_message(
    "asia-northeast1、us-central1、europe-west1 の p50 / p95 レイテンシを取得して比較し、"
    "東京のユーザー向けに最適なプライマリおよびフェイルオーバー先リージョンを推奨してください。"
)
print(recommendation.text)

print("\n--- 記録されたツール呼び出しトレース ---")
for turn in tool_chat.get_history():
    for part in turn.parts or []:
        if part.function_call:
            print(f"-> FunctionCall: {part.function_call.name}({dict(part.function_call.args or {})})")
        elif part.function_response:
            print(f"<- FunctionResponse: {part.function_response.response}")

最後に、全5章のクックブックの締めくくりとして、エージェントがツール経由で取得したマルチリージョンの `p50` / `p95` レイテンシデータを、`setup(..., cwd=True)` と **`BarChart`** + `show()` を使ってノートブックの出力セル内に直接チャートとして可視化します。

In [ ]:
from drawlib.canvas import clear, setup, show
from drawlib.charts.bar import BarChart
from drawlib.styles import Styles
from drawlib.text import text

clear()
setup(width=116, height=56, cwd=True)

regions = list(REGION_LATENCY_MS.keys())
p50_ms = [get_region_latency(r)["p50_ms"] for r in regions]
p95_ms = [get_region_latency(r)["p95_ms"] for r in regions]

text(
    (58.0, 50.8),
    "Gemini Function Calling で取得したリージョン別 API レイテンシ比較",
    style=Styles.DarkBold.patch(text_size=11.5),
)

chart = BarChart(
    axis_line_style=Styles.MutedDashed,
    categories=regions,
    width=96.0,
    height=34.0,
    orientation="vertical",
    bar_mode="group",
    bar_width_ratio=0.62,
    bar_r=0.8,
    axis_text_style=Styles.Dark.patch(text_size=10.0),
    grid_style=Styles.MutedThin,
    value_text_style=Styles.DarkBold.patch(text_size=10.0),
    value_format="{:.1f} ms",
)
chart.add_series("p50 中央値 (ms)", p50_ms, style=Styles.PrimaryFlat)
chart.add_series("p95 テール遅延 (ms)", p95_ms, style=Styles.SecondaryNeutral)
chart.configure_y_axis(min_value=0.0, max_value=100.0, tick_step=25.0, unit=" ms")
chart.draw(xy=(12.0, 6.0))
chart.draw_legend(
    xy=(34.5, 44.5),
    text_style=Styles.Dark.patch(text_size=10.0),
    orientation="horizontal",
)

show()

> **まとめ**: `client.chats.create()` によるマルチターン文脈管理、型付き Python 関数を `tools=[fn]` に渡すだけの自動 Function Calling、そして `setup(..., cwd=True)` + `show()` による Drawlib チャート描画を組み合わせることで、データ取得から意思決定・可視化までを1つのノートブック内でシームレスに完結できます。